# LHT?
Lysine-Histidine transporter 1입니다. 그거 줄여서 LHT인거예요. 식물에 존재하는 수송 단백질 중 하나인데, 여기에 knock out 돌연변이가 생긴 돌연변이체는 ACC(에틸렌 전구체)에 반응하지 않습니다. 

애기장대에는 해당 단백질 가족이 10까지 있습니다. 

본인 이걸로 논문도 썼음... 

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython 
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Biopython
from Bio import Entrez, SeqIO # 왼쪽: 일단 털어보자/오른쪽: 시퀀스 다루려면 필요합니다. 필수임. 
from Bio import AlignIO # 서열 분석해줄 친구
from Bio import Phylo # 트리 그릴라면 필요해요 
from Bio.Align import AlignInfo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
from Bio import Align

import io # 누구세요?
import subprocess # 서브 프로세스(이건 또 뭐여...)
from collections import defaultdict
import re

# 통계분석용
from scipy.stats import mannwhitneyu
from itertools import combinations

In [ ]:
# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# Entrez 접속 세팅
Entrez.email = "blackholekun@gmail.com"

# MUSCLE 경로
muscle_exe = "/opt/homebrew/bin/muscle" # 이거 경로 있어야 써요

# 서열 정보 가져오기

In [ ]:
def analyze_lht1():
    # 1. LHT1 (AT5G40780)의 Accession 번호로 데이터 가져오기 (예: NM_123443)
    # 여기서는 예시로 NC_003076.8 (Chr 5) 영역의 정보를 타겟팅하거나 간단히 검색합니다.
    handle = Entrez.efetch(db="nucleotide", id="NM_123443", rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    handle.close()

    print(f"--- LHT1 유전자 분석 결과 ---")
    print(f"ID: {record.id}")
    print(f"설명: {record.description}")
    print(f"서열 길이: {len(record.seq)} bp")
    
    # 2. GC Content 계산 (식물 유전자 특성 파악)
    gc_val = gc_fraction(record.seq) * 100
    print(f"GC 함량: {gc_val:.2f}%")

    # 3. 아미노산 서열로 번역 (Translation)
    protein_seq = record.seq.translate(to_stop=True)
    print(f"단백질 서열(앞 20자): {protein_seq[:20]}...")

if __name__ == "__main__":
    analyze_lht1()

## MSA
- 다른 식물종들과 MSA 해보기
- KAM은 살비아고 KML이 호접란입니다. 

In [ ]:
# MSA 하려면 일단 재료를 갖고와야 합니다. 
def fetch_homologs(gene_name="LHT1", count=10):
    # 1. 식물(viridiplantae) 중에서 LHT1 검색
    search_query = f"{gene_name}[Gene Name] AND viridiplantae[Orgn]"
    handle = Entrez.esearch(db="protein", term=search_query, retmax=count)
    record = Entrez.read(handle)
    ids = record["IdList"]
    
    # 2. 검색된 ID들로 서열 정보 한꺼번에 가져오기
    fetch_handle = Entrez.efetch(db="protein", id=ids, rettype="fasta", retmode="text")
    
    # 3. 파일로 저장 (이게 바로 MSA 준비 끝!)
    with open(f"{gene_name}_homologs.fasta", "w") as f:
        f.write(fetch_handle.read())
    
    print(f"창고 털기 완료! {len(ids)}개의 서열을 {gene_name}_homologs.fasta에 담았습니다.")

fetch_homologs("LHT1", 15)

In [ ]:
# FASTA 파일 확인
def list_up_species(fasta_file):
    species_list = []
    
    # FASTA 파일 읽기
    for record in SeqIO.parse(fasta_file, "fasta"):
        # 헤더에서 대괄호 [ ] 안에 있는 학명 추출 (NCBI 포맷 기준)
        match = re.search(r'\[(.*?)\]', record.description)
        species = match.group(1) if match else "Unknown"
        
        species_list.append({
            "Accession": record.id,
            "Species": species,
            "Length": len(record.seq),
            "Description": record.description[:50] + "..." # 너무 길면 생략
        })
    
    # 데이터프레임으로 변환
    df = pd.DataFrame(species_list)
    
    # 중복 제거 및 종별 카운트 확인
    print(f"총 {len(df)}개의 서열이 확인되었습니다.")
    print("\n--- 발견된 식물 종 리스트 ---")
    print(df[['Species', 'Accession']].groupby('Species').count())
    
    return df

# 실행 (위에서 만든 파일 이름 사용)
df_lht1 = list_up_species("LHT1_homologs.fasta")
print(df_lht1.head(10))

In [ ]:
def run_muscle(input_fasta, output_aln):
    # MUSCLE 실행 (설치된 경로에 따라 'muscle' 또는 'muscle3' 등 확인 필요)
    # 구버전 기준: muscle -in input.fasta -out output.aln
    # 신버전(v5) 기준: muscle -align input.fasta -output output.aln
    command = ["muscle", "-align", input_fasta, "-output", output_aln]
    
    try:
        subprocess.run(command, check=True)
        print(f"Alignment 완료: {output_aln}")
    except FileNotFoundError:
        print("MUSCLE이 PATH에 설정되어 있지 않습니다. 경로를 확인해주세요!")

# 실행
alignment = run_muscle("LHT1_homologs.fasta", "LHT1_aligned.aln")

In [ ]:
def view_msa(alignment_file, start=0, end=60):
    # 1. 정렬 파일 읽기
    alignment = AlignIO.read(alignment_file, "fasta")
    
    print(f"\n{'='*20} LHT1 MSA Result (Region: {start}-{end}) {'='*20}")
    
    # 2. 각 시퀀스별로 정해진 구간 출력
    for record in alignment:
        # ID를 15자로 맞추고, 뒤에 서열 표시
        label = record.id[:15].ljust(15)
        sequence_segment = record.seq[start:end]
        print(f"{label} : {sequence_segment}")
    
    print(f"{'='*60}\n")

view_msa("LHT1_aligned.aln", start=100, end=160)


In [ ]:
view_msa("LHT1_aligned.aln", start=250, end=350)

### Identity 파악 및 필터링

In [ ]:
def calculate_identity_fixed(ref_id_fragment, alignment_file):
    align = AlignIO.read(alignment_file, "fasta")
    
    # 1. 레퍼런스(애기장대 LHT1) 서열 찾기
    try:
        ref_record = [rec for rec in align if ref_id_fragment in rec.id][0]
        ref_seq_str = str(ref_record.seq)
        ref_len_no_gap = len(ref_seq_str.replace("-", ""))
    except IndexError:
        print(f"ID에 '{ref_id_fragment}'를 포함하는 서열을 찾을 수 없습니다.")
        return None
    
    results = []
    for rec in align:
        target_seq_str = str(rec.seq)
        
        # 2. 갭을 제외하고 일치하는 아미노산 개수 계산
        matches = sum(1 for a, b in zip(ref_seq_str, target_seq_str) 
                        if a == b and a != "-")
        
        # 3. 퍼센트 계산
        identity = (matches / ref_len_no_gap) * 100
        results.append({
            "Accession": rec.id[:15], 
            "Identity(%)": round(identity, 2)
        })
    
    # 데이터프레임으로 변환 및 정렬
    df = pd.DataFrame(results).sort_values("Identity(%)", ascending=False)
    return df

# 실행 (애기장대 LHT1의 Accession 일부인 Q9FKS8 입력)
df_result = calculate_identity_fixed("Q9FKS8", "LHT1_aligned.aln")
print(df_result)

In [ ]:
# 1. Identity 계산 (결과가 None인지 꼭 확인)
df_result = calculate_identity_fixed("Q9FKS8", "LHT1_aligned.aln")

if df_result is not None:
    # 2. 60% 이상인 '일하는 과학자들'의 데이터만 추출
    # Accession 이름이 잘릴 수 있으니 원본 alignment에서 ID를 매칭합니다.
    top_hits = df_result[df_result['Identity(%)'] > 60]['Accession'].tolist()
    
    alignment = AlignIO.read("LHT1_aligned.aln", "fasta")
    filtered_records = []
    
    for rec in alignment:
        # Accession이 상위 리스트에 포함되는지 확인
        if any(hit in rec.id for hit in top_hits):
            filtered_records.append(rec)
            
    if filtered_records:
        SeqIO.write(filtered_records, "LHT1_Final_Candidates.fasta", "fasta")
        print(f"✅ 필터링 성공: {len(filtered_records)}개의 정예 멤버(외자엽/쌍자엽 포함)가 저장되었습니다.")
    else:
        print("⚠️ 필터링된 서열이 없습니다. 조건을 확인해주세요.")
else:
    print("❌ 레퍼런스 서열을 찾지 못해 분석을 진행할 수 없습니다.")

## 쌍떡잎 vs 외떡잎

In [ ]:
# 정예 멤버들만 모인 파일을 다시 정렬합니다.
input_file = "LHT1_Final_Candidates.fasta"
output_file = "LHT1_Final_Candidates_aligned.aln"

# MUSCLE 실행 (버전에 따라 옵션이 다를 수 있으니 확인!)
# v5 기준: muscle -align input -output output
# v3 기준: muscle -in input -out output
command = ["muscle", "-align", input_file, "-output", output_file]

try:
    subprocess.run(command, check=True)
    print(f"✨ 정예 멤버 정렬 완료: {output_file}")
except Exception as e:
    print(f"❌ MUSCLE 실행 중 오류 발생: {e}")

### 히트맵

### Phylogenic tree

In [ ]:
# 1. 필터링된 데이터로 정렬 읽기 (이미 MUSCLE을 돌렸다고 가정)
final_aln = AlignIO.read("LHT1_Final_Candidates_aligned.aln", "fasta")

# 2. NJ 트리 생성
calculator = DistanceCalculator('blosum62')
dm = calculator.get_distance(final_aln)
constructor = DistanceTreeConstructor(calculator, 'nj')
final_tree = constructor.build_tree(final_aln)

# 3. NanumSquare 폰트로 시각화
fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(1, 1, 1)
Phylo.draw(final_tree, axes=ax, do_show=False, label_func=lambda n: str(n) if n.is_terminal() else "")

plt.title("LHT1 정예 멤버 계통수: 외자엽 vs 쌍자엽", fontsize=15)
plt.show()

- 음... MSA에서는 살비아랑 더 가까운걸로 나왔는데, 이렇게 보니 살비아나 호접란이나 그렇게 가깝진 않은 듯? 

# At Homologue

In [ ]:
def fetch_arath_lht_sequences():
    # NCBI에서 찾을 LHT 1-10의 Gene Symbol 또는 Accession 리스트
    # Arabidopsis LHT 패밀리의 대표적인 Accession 번호들입니다.
    lht_ids = [
        "NP_199343.1", # LHT1 (At5g40780)
        "NP_174246.1", # LHT2 (At1g24360)
        "NP_174245.1", # LHT3 (At1g24350)
        "NP_195514.1", # LHT4 (At1g47670)
        "NP_195513.1", # LHT5 (At1g47660)
        "NP_188049.1", # LHT6 (At3g13560)
        "NP_191295.1", # LHT7 (At3g57350)
        "NP_565985.1", # LHT8 (At2g41780)
        "NP_191296.1", # LHT9 (At3g57360)
        "NP_200159.1"  # LHT10 (At5g53460)
    ]
    
    print("🚀 NCBI에서 LHT 1-10 서열을 가져오는 중...")
    records = []
    for i, acc in enumerate(lht_ids):
        handle = Entrez.efetch(db="protein", id=acc, rettype="fasta", retmode="text")
        record = SeqIO.read(handle, "fasta")
        # 이름을 알아보기 쉽게 LHT1, LHT2... 형식으로 변경
        record.id = f"LHT{i+1}_ARATH_{acc}"
        records.append(record)
        handle.close()
    
    # FASTA 파일로 저장
    SeqIO.write(records, "ARATH_LHT_1_10.fasta", "fasta")
    print("✅ ARATH_LHT_1_10.fasta 저장 완료.")
    return records

# 2. 분석 실행부
# 서열 확인
records = fetch_arath_lht_sequences()
for r in records:
    print(f"{r.id}: {len(r.seq)} aa")

# 3. MSA (MUSCLE 사용 - 로컬에 muscle.exe가 있어야 함)
# 만약 로컬에 muscle이 없다면 Clustal 등을 사용하거나 온라인 정렬 도구를 써야 합니다.
# 여기서는 로직만 작성합니다.
def run_msa(input_fasta, output_aln):
    print("🧬 최신 MUSCLE 명령어로 MSA 진행 중...")
    
    # MUSCLE v5 이상의 표준 명령어 형식입니다.
    # 만약 여전히 에러가 난다면 'muscle' 대신 'muscle5' 등으로 이름을 확인해보세요.
    cmd = ["muscle", "-align", input_fasta, "-output", output_aln]
    
    try:
        # Biopython wrapper 대신 파이썬 표준 라이브러리로 직접 실행
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"✅ MSA 완료: {output_aln}")
        else:
            # v5 명령어가 안 먹힐 경우를 대비한 v3 명령어 재시도
            print("🔄 구버전 명령어로 재시도 중...")
            cmd_old = ["muscle", "-in", input_fasta, "-out", output_aln]
            result_old = subprocess.run(cmd_old, capture_output=True, text=True)
            
            if result_old.returncode == 0:
                print(f"✅ MSA 완료 (v3 방식): {output_aln}")
            else:
                print("❌ MSA 실패:")
                print(result_old.stderr)
                
    except FileNotFoundError:
        print("❌ 에러: 시스템에서 'muscle' 실행 파일을 찾을 수 없습니다.")
        print("환경 변수(Path)에 MUSCLE이 등록되어 있는지 확인해 주세요.")

run_msa("ARATH_LHT_1_10.fasta", "ARATH_LHT_aligned.aln")

In [ ]:
def view_msa(alignment_file, start=0, end=60):
    # 1. 정렬 파일 읽기
    alignment = AlignIO.read(alignment_file, "fasta")
    
    print(f"\n{'='*20} LHT1 MSA Result (Region: {start}-{end}) {'='*20}")
    
    # 2. 각 시퀀스별로 정해진 구간 출력
    for record in alignment:
        # ID를 15자로 맞추고, 뒤에 서열 표시
        label = record.id[:15].ljust(15)
        sequence_segment = record.seq[start:end]
        print(f"{label} : {sequence_segment}")
    
    print(f"{'='*60}\n")

view_msa("ARATH_LHT_aligned.aln", start=200, end=300)

In [ ]:
def plot_lht_family_tree(alignment_file):
    align = AlignIO.read(alignment_file, "fasta")
    calculator = DistanceCalculator('identity')
    constructor = DistanceTreeConstructor(calculator, 'nj')
    tree = constructor.build_tree(align)
    
    fig = plt.figure(figsize=(15, 8))
    ax = fig.add_subplot(1, 1, 1)
    plt.title("Arabidopsis LHT Family (1-10) Phylogeny (NCBI Data)", fontsize=15)
    
    # 연구자님이 설정한 전역 폰트(NanumSquare)가 적용됩니다.
    Phylo.draw(tree, axes=ax, do_show=False, label_func=lambda n: str(n) if n.is_terminal() else "")
    plt.tight_layout()
    plt.show()

plot_lht_family_tree("ARATH_LHT_aligned.aln")

- 내가 저거 확인한다고 외장하드 뒤져서 논문 서플까지 확인했다... 

In [ ]:
def plot_flexible_heatmap(alignment_file, target_id):
    """
    alignment_file: MSA 결과 파일 (.aln)
    target_id: 기준이 될 유전자 ID (예: 'LHT1' 또는 'At5g40780')
    """
    align = AlignIO.read(alignment_file, "fasta")
    
    # 1. 매개변수로 받은 target_id가 포함된 레코드 찾기
    target_idx = -1
    for i, rec in enumerate(align):
        if target_id in rec.id:
            target_idx = i
            break
            
    if target_idx == -1:
        print(f"❌ 오류: 파일 내에서 '{target_id}'를 찾을 수 없습니다.")
        return

    target_seq = str(align[target_idx].seq)
    names = [rec.id for rec in align]
    identities = []

    # 2. 기준 서열 vs 전체 서열 비교 (Gap-corrected)
    for rec in align:
        curr_seq = str(rec.seq)
        matches = sum(1 for a, b in zip(target_seq, curr_seq) if a == b and a != "-")
        valid_len = sum(1 for a, b in zip(target_seq, curr_seq) if a != "-" and b != "-")
        
        identity = (matches / valid_len * 100) if valid_len > 0 else 0
        identities.append(identity)

    # 3. 데이터프레임 및 시각화
    df = pd.DataFrame(identities, index=names, columns=[f'Standard: {target_id}'])
    df = df.sort_values(by=df.columns[0], ascending=False)

    plt.figure(figsize=(8, 10))
    sns.heatmap(df, annot=True, fmt=".1f", cmap="Blues")
    plt.title(f"Comparison based on {target_id}")
    plt.tight_layout()
    return df

plot_flexible_heatmap("ARATH_LHT_aligned.aln", "LHT1")

- ? 논문이랑 많이 다른데? 논문에 있는 염색체 번호로 다시 찾아보겠음. 

In [ ]:
def fetch_sequences_by_locus():
    # 연구자님이 주신 논문 기반 Locus ID 매핑
    locus_map = {
        "LHT1": "At5g40780", "LHT2": "At1g24400", "LHT3": "At1g61270",
        "LHT4": "At1g47670", "LHT5": "At1g67640", "LHT6": "At3g01760",
        "LHT7": "At4g36180", "LHT8": "At1g71680", "LHT9": "At1g48640",
        "LHT10": "At1g25530"
    }
    
    records = []
    print("🚀 논문 Locus ID 기준으로 진짜 서열 소환 중...")
    
    for name, locus in locus_map.items():
        # Locus ID로 검색하여 가장 신뢰도 높은 RefSeq(NP_) 혹은 유추 서열 가져오기
        search_handle = Entrez.esearch(db="protein", term=f"{locus}[All Fields] AND Arabidopsis thaliana[Organism]")
        search_results = Entrez.read(search_handle)
        search_handle.close()
        
        if search_results["IdList"]:
            prot_id = search_results["IdList"][0]
            fetch_handle = Entrez.efetch(db="protein", id=prot_id, rettype="fasta", retmode="text")
            record = SeqIO.read(fetch_handle, "fasta")
            record.id = f"{name}_{locus}"
            records.append(record)
            fetch_handle.close()
            print(f"✅ {name} ({locus}) 확보 완료")
            
    SeqIO.write(records, "LHT_Paper_Original.fasta", "fasta")
    return "LHT_Paper_Original.fasta"

# 실행
fasta_file = fetch_sequences_by_locus()

In [ ]:
run_msa("LHT_Paper_Original.fasta", "LHT_Paper_Original.aln")

In [ ]:
view_msa("LHT_Paper_Original.aln", start=150, end=300)

In [ ]:
plot_lht_family_tree("LHT_Paper_Original.aln")

In [ ]:
plot_flexible_heatmap("LHT_Paper_Original.aln", "At5g40780")

- 논문하고 좀 달라졌다. 그동안 뭐 업뎃되고 해서 그런가...
- 그 와중에 LHT7 쟤는 뭐 방계임? 

### 히트맵 나란히 보기

In [ ]:
def draw_heatmap_to_ax(alignment_file, target_id, ax, title):
    """
    ax: plt.subplot에서 생성된 해당 구역을 인자로 받음
    """
    try:
        align = AlignIO.read(alignment_file, "fasta")
    except Exception as e:
        ax.text(0.5, 0.5, f"파일 없음:\n{alignment_file}", ha='center')
        return

    # 1. 기준점 찾기
    target_idx = -1
    for i, rec in enumerate(align):
        if target_id in rec.id:
            target_idx = i
            break
            
    if target_idx == -1:
        ax.text(0.5, 0.5, f"ID 못 찾음:\n{target_id}", ha='center')
        return

    target_seq = str(align[target_idx].seq)
    names = [rec.id.split('_')[0] for rec in align] # 이름 간소화
    identities = []

    # 2. 유사도 계산
    for rec in align:
        curr_seq = str(rec.seq)
        matches = sum(1 for a, b in zip(target_seq, curr_seq) if a == b and a != "-")
        valid_len = sum(1 for a, b in zip(target_seq, curr_seq) if a != "-" and b != "-")
        identities.append((matches / valid_len * 100) if valid_len > 0 else 0)

    # 3. 데이터프레임 구성
    df = pd.DataFrame(identities, index=names, columns=['Identity (%)'])
    df = df.sort_values(by='Identity (%)', ascending=False)

    # 4. 지정된 ax에 히트맵 그리기
    sns.heatmap(df, annot=True, fmt=".1f", cmap="Blues", ax=ax, cbar=(ax.get_subplotspec().is_last_col()))
    ax.set_title(title)

# --- 실행 부분 ---
plt.figure(figsize=(14, 10))

# 왼쪽: NCBI 데이터
ax1 = plt.subplot(1, 2, 1)
draw_heatmap_to_ax("ARATH_LHT_aligned.aln", "LHT1", ax1, "NCBI (LHT1 기준)")

# 오른쪽: 논문 데이터 (염색체 번호 기반)
ax2 = plt.subplot(1, 2, 2)
draw_heatmap_to_ax("LHT_Paper_Original.aln", "At5g40780", ax2, "Paper (At5g40780 기준)")

plt.tight_layout()
plt.show()

In [ ]:
def map_by_locus_id(ncbi_fasta, paper_fasta):
    # 1. 서열 읽기
    ncbi_recs = list(SeqIO.parse(ncbi_fasta, "fasta"))
    paper_recs = list(SeqIO.parse(paper_fasta, "fasta"))

    mapping = []
    
    # 2. 정규표현식으로 At#g##### 패턴 추출 함수
    def extract_locus(text):
        match = re.search(r'At[1-5]g\d{5}', text, re.IGNORECASE)
        return match.group(0).upper() if match else None

    for p_rec in paper_recs:
        p_locus = extract_locus(p_rec.id)
        
        # NCBI 서열 중 이 Locus ID를 가진 녀석이 있는지 확인
        match_found = False
        for n_rec in ncbi_recs:
            n_locus = extract_locus(n_rec.description) # description까지 확인
            
            if p_locus and n_locus and p_locus == n_locus:
                mapping.append({
                    "Paper_Name": p_rec.id.split('_')[0],
                    "Locus_ID": p_locus,
                    "NCBI_Original_Name": n_rec.id,
                    "Match_Status": "🎯 정확히 찾음"
                })
                match_found = True
                break
        
        if not match_found:
            mapping.append({
                "Paper_Name": p_rec.id.split('_')[0],
                "Locus_ID": p_locus,
                "NCBI_Original_Name": "❌ NCBI 리스트에 없음",
                "Match_Status": "데이터 누락"
            })

    df = pd.DataFrame(mapping)
    print("\n[염색체 번호 기준 신원 확인 결과]")
    print(df.to_string(index=False))
    return df

# 실행
final_mapping = map_by_locus_id("ARATH_LHT_1_10.fasta", "LHT_Paper_Original.fasta")

- 10년 전에도 그러더니 아직도 하나도 안 맞는단 말이냐... 